In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
import json
import math
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import logging
from datetime import datetime
import numpy as np
from sklearn.metrics import average_precision_score, roc_auc_score, roc_curve
import random

In [ ]:
def serialize_message(messages):
  parts = []
  for m in messages:
    role = m["role"].upper()
    content = m["content"].strip()
    parts.append(f"[{role}] {content}")
  return "\n".join(parts)

In [ ]:
class EncoderBinaryClassifier(nn.Module):
  def __init__(self, encoder_name="roberta-base", dropout=0.1):
    super().__init__()
    self.encoder = AutoModel.from_pretrained(encoder_name)
    hidden_size = self.encoder.config.hidden_size
    self.dropout = nn.Dropout(dropout)
    self.classifier = nn.Linear(hidden_size, 1)

  def forward(self, input_ids, attention_mask):
    outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
    cls = outputs.last_hidden_state[:, 0, :]
    cls = self.dropout(cls)
    logits = self.classifier(cls).squeeze(-1)
    return logits

In [ ]:
class SafetyDataset(Dataset):
  def __init__(self, data):
    self.texts = [serialize_message(ex["messages"]) for ex in data]
    self.labels = [float(ex["label"]) for ex in data]

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return {"text": self.texts[idx], "label": self.labels[idx]}

In [ ]:
def make_collate_fn(tokenizer, max_length=512, stride=128):
  def collate(batch):
    texts = [x["text"] for x in batch]
    labels = torch.tensor([x["label"] for x in batch], dtype=torch.float32)

    enc = tokenizer(
      texts,
      padding=True,
      truncation=True,
      max_length=max_length,
      stride=stride,
      return_overflowing_tokens=True,
      return_tensors="pt"
    )

    sample_map = enc.pop("overflow_to_sample_mapping").long()
    enc["labels"] = labels
    enc["sample_map"] = sample_map
    return enc
  return collate

In [ ]:
def aggregate_chunk_logits(chunk_logits, sample_map, batch_size):
  out = torch.full((batch_size,), -1e9, device=chunk_logits.device)
  out.scatter_reduce_(0, sample_map, chunk_logits, reduce="amax", include_self=True)
  return out

In [ ]:
@torch.no_grad()
def validate_loss(model, val_loader, device):
  model.eval()
  loss_fn = nn.BCEWithLogitsLoss()
  total_loss = 0.0
  n = 0
  for batch in val_loader:
    sample_map = batch.pop("sample_map").to(device)
    labels = batch.pop("labels").to(device)
    B = labels.size(0)
    batch = {k: v.to(device) for k, v in batch.items()}
    chunk_logits = model(**batch)
    ex_logits = aggregate_chunk_logits(chunk_logits, sample_map, B)
    loss = loss_fn(ex_logits, labels)
    bs = labels.size(0)
    total_loss += float(loss.item()) * bs
    n += bs
  return total_loss / max(1, n)

In [ ]:
def train_with_val(
  model,
  tokenizer,
  train_data,
  val_data,
  device=None,
  batch_size=16,
  val_batch_size=32,
  max_length=512,
  epochs=5,
  lr=2e-5,
  weight_decay=0.01,
  max_grad_norm=1.0,
  val_every_steps=200,
):
  for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)

  log_filename = f"training_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"

  logging.basicConfig(
      filename=log_filename,
      filemode="w",
      level=logging.INFO,
      format="%(asctime)s | %(message)s",
      force=True,
  )

  logging.info("Starting training")
  print("Starting training")

  device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
  model.to(device)

  train_ds = SafetyDataset(train_data)
  val_ds = SafetyDataset(val_data)

  collate = make_collate_fn(tokenizer, max_length=max_length)

  train_loader = DataLoader(
    train_ds, batch_size=batch_size, shuffle=True, num_workers=2, collate_fn=collate
  )
  val_loader = DataLoader(
    val_ds, batch_size=val_batch_size, shuffle=False, num_workers=2, collate_fn=collate
  )

  loss_fn = nn.BCEWithLogitsLoss()
  optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

  best_val = float("inf")
  global_step = 0

  for epoch in range(1, epochs + 1):
    model.train()
    running_loss = 0.0
    seen = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{epochs}", leave=False)

    for batch in pbar:
      global_step += 1

      sample_map = batch.pop("sample_map").to(device)
      labels = batch.pop("labels").to(device)
      B = labels.size(0)
      batch = {k: v.to(device) for k, v in batch.items()}

      chunk_logits = model(**batch)
      ex_logits = aggregate_chunk_logits(chunk_logits, sample_map, B)
      loss = loss_fn(ex_logits, labels)

      optimizer.zero_grad(set_to_none=True)
      loss.backward()
      torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
      optimizer.step()

      bs = labels.size(0)
      running_loss += float(loss.item()) * bs
      seen += bs

      avg_loss = running_loss / max(1, seen)
      pbar.set_postfix(train_loss=f"{avg_loss:.4f}")

      if global_step % val_every_steps == 0:
        val_loss = validate_loss(model, val_loader, device)
        logging.info(f"\nepoch={epoch} step={global_step} train_loss={avg_loss:.4f} val_loss={val_loss:.4f}")
        print(f"\nepoch={epoch} step={global_step} train_loss={avg_loss:.4f} val_loss={val_loss:.4f}")

        if val_loss < best_val:
          best_val = val_loss
          torch.save(model.state_dict(), "best_model.pt")
          logging.info("saved best_model.pt")
          print("saved best_model.pt")

        model.train()

    train_loss = running_loss / max(1, seen)
    val_loss = validate_loss(model, val_loader, device)
    model.train()

    logging.info(f"\nepoch_end={epoch} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")
    print(f"\nepoch_end={epoch} train_loss={train_loss:.4f} val_loss={val_loss:.4f}")

    ckpt_path = f"checkpoint_epoch_{epoch}.pt"
    torch.save(model.state_dict(), ckpt_path)
    logging.info(f"saved {ckpt_path}")
    print(f"saved {ckpt_path}")

    if val_loss < best_val:
      best_val = val_loss
      torch.save(model.state_dict(), "best_model.pt")
      logging.info("saved best_model.pt")
      print("saved best_model.pt")

  logging.info(f"\ndone. best_val_loss={best_val:.4f}")
  print(f"\ndone. best_val_loss={best_val:.4f}")
  return model

In [ ]:
encoder_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(encoder_name)

with open("/content/train_final.json", "r") as f:
    train_data = json.load(f)

with open("/content/val_final.json", "r") as f:
    val_data = json.load(f)

model = EncoderBinaryClassifier(encoder_name=encoder_name, dropout=0.1)

train_with_val(
  model=model,
  tokenizer=tokenizer,
  train_data=train_data,
  val_data=val_data,
  batch_size=16,
  val_batch_size=32,
  epochs=3,
  lr=2e-5,
  val_every_steps=200,
)

In [ ]:
DATA_PATH = "/content/eval.json"
CKPT_PATH = "/content/checkpoint_epoch_2.pt"
ENCODER_NAME = "roberta-base"
BATCH_SIZE = 32
MAX_LENGTH = 512
DROPOUT = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

with open(DATA_PATH, "r") as f:
    data = json.load(f)

y_true = np.array([int(ex["label"]) for ex in data], dtype=np.int32)

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)

model = EncoderBinaryClassifier(encoder_name=ENCODER_NAME, dropout=DROPOUT)
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state)
model.to(device)
model.eval()

# ---- PREDICT ----
@torch.no_grad()
def predict_probs(examples):
    probs = []
    for i in range(0, len(examples), BATCH_SIZE):
        batch = examples[i:i+BATCH_SIZE]
        texts = [serialize_message(ex["messages"]) for ex in batch]

        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(enc["input_ids"], enc["attention_mask"])
        p = torch.sigmoid(logits).detach().cpu().numpy()
        probs.append(p)

    return np.concatenate(probs, axis=0)

y_score = predict_probs(data)

# ---- METRICS ----
aupr = float(average_precision_score(y_true, y_score))
rocauc = float(roc_auc_score(y_true, y_score))

fpr, tpr, thr = roc_curve(y_true, y_score)

def fpr_at_tpr(target_tpr):
    idx = np.where(tpr >= target_tpr)[0]
    return float("nan") if len(idx) == 0 else float(fpr[idx[0]])

fpr90 = fpr_at_tpr(0.90)
fpr95 = fpr_at_tpr(0.95)

print(f"AUPR: {aupr:.6f}")
print(f"ROC-AUC: {rocauc:.6f}")
print(f"FPR @ 90% TPR: {fpr90:.6f}")
print(f"FPR @ 95% TPR: {fpr95:.6f}")

In [ ]:
DATA_PATH = "/content/eval.json"
CKPT_PATH = "/content/checkpoint_epoch_2.pt"
ENCODER_NAME = "roberta-base"
BATCH_SIZE = 32
MAX_LENGTH = 512
DROPOUT = 0.1

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

with open(DATA_PATH, "r") as f:
    data = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)

model = EncoderBinaryClassifier(encoder_name=ENCODER_NAME, dropout=DROPOUT)
state = torch.load(CKPT_PATH, map_location="cpu")
model.load_state_dict(state)
model.to(device)
model.eval()

@torch.no_grad()
def score_examples(examples):
    scores = []
    for i in range(0, len(examples), BATCH_SIZE):
        batch = examples[i:i+BATCH_SIZE]
        texts = [serialize_message(ex["messages"]) for ex in batch]

        enc = tokenizer(
            texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        enc = {k: v.to(device) for k, v in enc.items()}

        logits = model(enc["input_ids"], enc["attention_mask"])
        probs = torch.sigmoid(logits).detach().cpu().numpy()
        scores.append(probs)

    return np.concatenate(scores, axis=0)

# ---- SCORE ALL ----
y_score = score_examples(data)

# ---- FILTER BENIGN ONLY ----
benign_indices = [i for i, ex in enumerate(data) if int(ex["label"]) == 0]
benign_scores = [(i, y_score[i]) for i in benign_indices]

# sort by predicted harmful probability descending
benign_scores.sort(key=lambda x: -x[1])

top10 = benign_scores[:25]

print("\n===== TOP 10 FALSE POSITIVE CANDIDATES =====\n")

for rank, (idx, score) in enumerate(top10, 1):
    ex = data[idx]
    print(f"Rank {rank}")
    print(f"Predicted harmful probability: {float(score):.4f}")
    print(f"True label: {ex['label']}")
    print("Messages:")
    print(serialize_message(ex["messages"]))
    print("-" * 80)

In [ ]:
y_score_1d = np.asarray(y_score).reshape(-1)

benign_indices = [i for i, ex in enumerate(data) if int(ex["label"]) == 0]
harmful_indices = [i for i, ex in enumerate(data) if int(ex["label"]) == 1]

sampled_benign = random.sample(benign_indices, 5)
sampled_harmful = random.sample(harmful_indices, 5)

print("\n===== 5 RANDOM BENIGN + 5 RANDOM HARMFUL =====\n")

all_indices = sampled_benign + sampled_harmful

for i, idx in enumerate(all_indices, 1):
    ex = data[idx]
    prob = float(y_score_1d[idx])
    label = ex["label"]

    print(f"Sample {i}")
    print(f"Predicted harmful probability: {prob:.6f}")
    print(f"True label: {label}")
    print("Messages:")
    print(serialize_message(ex["messages"]))
    print("-" * 80)

In [ ]:
y_score_1d = np.asarray(y_score).reshape(-1)
fpr, tpr, thr = roc_curve(y_true, y_score_1d)

def threshold_at_tpr(target_tpr):
    idx = np.where(tpr >= target_tpr)[0]
    if len(idx) == 0:
        return None
    j = idx[0]
    return float(thr[j]), float(fpr[j]), float(tpr[j])

th90, f90, t90 = threshold_at_tpr(0.90)
th95, f95, t95 = threshold_at_tpr(0.95)

print("Threshold @ 90% TPR:", th90)
print("Threshold @ 95% TPR:", th95)